In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("online_retail_cleaned.csv")

In [5]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [7]:
df.shape

(805549, 9)

In [9]:
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

print("Reference date we're using:", reference_date)

Reference date we're using: 2011-12-10 12:50:00


In [15]:
# We group all transactions by Customer ID
# Then for each customer, we calculate their R, F, and M values

rfm = df.groupby("Customer ID").agg(
    Recency   = ("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency = ("Invoice", "nunique"),
    Monetary  = ("Revenue", "sum")
).reset_index()

print("Number of unique customers:", len(rfm))
rfm.head(10)

Number of unique customers: 5878


,Customer ID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,5633.32
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40
5,12351.0,375,1,300.93
6,12352.0,36,10,2849.84
7,12353.0,204,2,406.76
8,12354.0,232,1,1079.40
9,12355.0,214,2,947.61


In [17]:
# give each customer a score of 1-4 on each dimension

# RECENCY: lower days = better = higher score
rfm["R_Score"] = pd.qcut(rfm["Recency"], q=4, labels=[4, 3, 2, 1])

# FREQUENCY: higher = better = higher score
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), q=4, labels=[1, 2, 3, 4])

# MONETARY: higher = better = higher score
rfm["M_Score"] = pd.qcut(rfm["Monetary"], q=4, labels=[1, 2, 3, 4])

rfm["R_Score"] = rfm["R_Score"].astype(int)
rfm["F_Score"] = rfm["F_Score"].astype(int)
rfm["M_Score"] = rfm["M_Score"].astype(int)

rfm.head(10)

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,12346.0,326,12,77556.46,2,4,4
1,12347.0,2,8,5633.32,4,4,4
2,12348.0,75,5,2019.40,3,3,3
3,12349.0,19,4,4428.69,4,3,4
4,12350.0,310,1,334.40,2,1,1
5,12351.0,375,1,300.93,2,1,1
6,12352.0,36,10,2849.84,3,4,4
7,12353.0,204,2,406.76,2,2,2
8,12354.0,232,1,1079.40,2,1,3
9,12355.0,214,2,947.61,2,2,3


In [19]:
# Combine the 3 scores into one number

rfm["RFM_Score"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

print("Score range:", rfm["RFM_Score"].min(), "to", rfm["RFM_Score"].max())
print(rfm["RFM_Score"].value_counts().sort_index())

Score range: 3 to 12
RFM_Score
3     575
4     572
5     583
6     629
7     591
8     622
9     568
10    556
11    521
12    661
Name: count, dtype: int64


In [23]:
# use the scores to put each customer into a named group

def assign_segment(row):
    r = row["R_Score"]
    f = row["F_Score"]
    m = row["M_Score"]
    score = row["RFM_Score"]
    
    if r >= 4 and f >= 3 and m >= 3:
        return "Champion"
    elif r >= 3 and f >= 3:
        return "Loyal Customer"
    elif r >= 3 and f <= 2:
        return "Potential Loyalist"
    elif r == 2 and f >= 2:
        return "At Risk"
    elif r == 2 and f == 1:
        return "About to Sleep"
    else:
        return "Lost"

rfm["Segment"] = rfm.apply(assign_segment, axis=1)

rfm["Segment"].value_counts()

Segment
Lost                  1459
At Risk               1104
Champion              1045
Loyal Customer        1014
Potential Loyalist     894
About to Sleep         362
Name: count, dtype: int64

In [25]:
rfm.to_csv("rfm_output.csv", index=False)

rfm.head(10)

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,12346.0,326,12,77556.46,2,4,4,10,At Risk
1,12347.0,2,8,5633.32,4,4,4,12,Champion
2,12348.0,75,5,2019.40,3,3,3,9,Loyal Customer
3,12349.0,19,4,4428.69,4,3,4,11,Champion
4,12350.0,310,1,334.40,2,1,1,4,About to Sleep
5,12351.0,375,1,300.93,2,1,1,4,About to Sleep
6,12352.0,36,10,2849.84,3,4,4,11,Loyal Customer
7,12353.0,204,2,406.76,2,2,2,6,At Risk
8,12354.0,232,1,1079.40,2,1,3,6,About to Sleep
9,12355.0,214,2,947.61,2,2,3,7,At Risk
